## Imports

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

## Configuration

In [ ]:
IAM_DIR = "iam"
IMG_H, IMG_W = 64, 256
BATCH_SIZE = 64
EPOCHS = 40
SEED = 42

## Unpack data files

In [ ]:
!mkdir -p /content/iam/words
!tar -xzf /content/words.tgz -C /content/iam/words
!tar -xzf /content/ascii.tgz -C /content/iam

## Load data files

In [ ]:
# Loads dataset into samples
def load_iam_words(iam_dir):
    samples = []
    with open(os.path.join(iam_dir, "words.txt")) as f:
        for line in f:
            # Skip comments
            if line.startswith("#"):
                continue
            parts = line.strip().split(" ")
            word_id, status = parts[0], parts[1]
            if status == "err":
                continue
            text = " ".join(parts[8:])
            pieces = word_id.split("-")
            path = os.path.join(
                iam_dir, "words", pieces[0], f"{pieces[0]}-{pieces[1]}", f"{word_id}.png"
            )
            if os.path.exists(path):
                samples.append((path, text))
    return samples

In [ ]:
# Load IAM dataset
samples = load_iam_words(IAM_DIR)

## Filter data files

In [ ]:
# Returns true if image can be decoded as grayscale
def decodes_ok(path):
    try:
        tf.io.decode_png(tf.io.read_file(path), channels=1)
        return True
    except tf.errors.InvalidArgumentError:
        return False

In [ ]:
# Keeps only images that decode
good = []
for p, t in samples:
    if decodes_ok(p):
        good.append((p, t))
    else:
        print("Skipping bad image:", p)
samples = good
print("Usable images:", len(samples))

## Split data files

In [ ]:
# Shuffles samples randomly using seed
random.Random(SEED).shuffle(samples)

# Split sample 90% train, 5% validation, 5% test
n = len(samples)
train_s = samples[: int(0.9 * n)]
val_s = samples[int(0.9 * n): int(0.95 * n)]
test_s = samples[int(0.95 * n):]

## Extract characters


In [ ]:
# Alphabet comes from the training transcriptions only
chars = sorted({c for _, t in train_s for c in t})
char_to_num = {c: i for i, c in enumerate(chars)}
num_to_char = {i: c for c, i in char_to_num.items()}
NUM_CLASSES = len(chars) + 1         # last index = CTC blank
MAX_LEN = max(len(t) for _, t in samples)
print("Characters:", "".join(chars))
# print("Num classes (incl. blank):", NUM_CLASSES, "| longest word:", MAX_LEN)
json.dump(chars, open("/content/chars.json", "w"))

## Encode labels

In [ ]:
# Encode transcripts as 0 padded lists of integers in labels
def encode_split(split):
    paths = [p for p, _ in split]                                                 # Image paths
    encoded = [[char_to_num[c] for c in t if c in char_to_num] for _, t in split]
    lengths = np.array([len(e) for e in encoded], dtype="int32")                  # Transcript lengths
    labels = np.zeros((len(split), MAX_LEN), dtype="int32")                       # Encoded labels
    for i, e in enumerate(encoded):
        labels[i, : len(e)] = e
    return paths, labels, lengths

## Preprocess images

In [ ]:
# Preprocess image as a float tensor where ink=1 and paper=0
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)   # 0 to 1
    img = 1.0 - img                                       # ink = 1 paper = 0
    img = tf.image.resize_with_pad(img, IMG_H, IMG_W)     # Aspect ratio
    return img

## Create dataset

In [ ]:
# Groups images, labels and lengths into datasets for training
def make_dataset(split, training):
    paths, labels, lengths = encode_split(split)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, lengths)) # (path, label, length)
    if training:
        ds = ds.shuffle(10_000, seed=SEED)

    def _map(p, l, n):
        return {
            "image": load_image(p),
            "label": l,
            "label_length": tf.reshape(n, [1]),
        }

    return ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create training, validation, and test datasets
train_ds = make_dataset(train_s, training=True)
val_ds = make_dataset(val_s, training=False)
test_ds = make_dataset(test_s, training=False)

## CTC Loss

In [ ]:
# Calculates CTC loss to align character predictions with image regions
class CTCLoss(layers.Layer):
    def call(self, labels, logits, label_length):
        batch = tf.shape(logits)[0]
        time_steps = tf.shape(logits)[1]
        loss = tf.nn.ctc_loss(
            labels=tf.cast(labels, tf.int32),
            logits=logits,
            label_length=tf.cast(tf.reshape(label_length, [-1]), tf.int32),
            logit_length=tf.fill([batch], time_steps),
            logits_time_major=False,
            blank_index=-1,
        )
        self.add_loss(tf.reduce_mean(loss))
        return logits

## CNN

In [ ]:
# Extracts visual features and reduces the image dimensions.
def conv_block(x, filters, pool):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return layers.MaxPooling2D(pool)(x)

## Build models

In [ ]:
# Builds the training model and the separate inference model
def build_models():
    image = layers.Input(shape=(IMG_H, IMG_W, 1), name="image")
    label = layers.Input(shape=(MAX_LEN,), dtype="int32", name="label")
    label_length = layers.Input(shape=(1,), dtype="int32", name="label_length")

    x = conv_block(image, 32, (2, 2))       # 32 x 128
    x = conv_block(x, 64, (2, 2))           # 16 x 64
    x = conv_block(x, 128, (2, 1))          #  8 x 64

    x = layers.Permute((2, 1, 3))(x)
    x = layers.Reshape((IMG_W // 4, (IMG_H // 8) * 128))(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    logits = layers.Dense(NUM_CLASSES, name="logits")(x)

    train_out = CTCLoss()(label, logits, label_length)
    train_model = models.Model([image, label, label_length], train_out, name="train_model")
    infer_model = models.Model(image, logits, name="infer_model")
    return train_model, infer_model

train_model, infer_model = build_models()
infer_model.summary()

## Decode predictions


In [ ]:
# Converts model predictions into readable character sequences
def decode_logits(logits):
    logits = tf.convert_to_tensor(logits)
    batch, time_steps = tf.shape(logits)[0], tf.shape(logits)[1]
    decoded, _ = tf.nn.ctc_greedy_decoder(
        tf.transpose(logits, [1, 0, 2]), tf.fill([batch], time_steps)
    )
    dense = tf.sparse.to_dense(decoded[0], default_value=-1).numpy()
    return ["".join(num_to_char[i] for i in row if i >= 0) for row in dense], decoded[0]

## Calculate CER

In [ ]:
# Calculates the average character error rate for a batch
def batch_cer(decoded_sparse, labels, lengths):
    """Character error rate for one batch (edit distance / label length)."""
    truth = tf.RaggedTensor.from_tensor(
        tf.cast(labels, tf.int64), lengths=tf.cast(tf.reshape(lengths, [-1]), tf.int64)
    ).to_sparse()
    dist = tf.edit_distance(tf.cast(decoded_sparse, tf.int64), truth, normalize=True)
    return float(tf.reduce_mean(dist))

## Train the model

In [ ]:
# Displays sample predictions and validation accuracy after each epoch
class ShowProgress(tf.keras.callbacks.Callback):
    def __init__(self, dataset, n_batches=10):
        super().__init__()
        self.dataset, self.n_batches = dataset, n_batches

    def on_epoch_end(self, epoch, logs=None):
        cers, shown = [], False
        for batch in self.dataset.take(self.n_batches):
            preds = infer_model(batch["image"], training=False)
            texts, sparse = decode_logits(preds)
            cers.append(batch_cer(sparse, batch["label"], batch["label_length"]))
            if not shown:
                for i in range(3):
                    truth = "".join(num_to_char[j] for j in batch["label"][i, : int(batch["label_length"][i, 0])].numpy())
                    print(f"   true: {truth!r:20} pred: {texts[i]!r}")
                shown = True
        print(f"   validation CER (sample): {np.mean(cers):.3f}")


In [ ]:
train_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3))

history = train_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        ShowProgress(val_ds),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ],
)

# Final test-set character error rate
cers = []
for batch in test_ds:
    _, sparse = decode_logits(infer_model(batch["image"], training=False))
    cers.append(batch_cer(sparse, batch["label"], batch["label_length"]))
print(f"Test CER: {np.mean(cers):.3f}")

infer_model.save("htr_infer_model.keras")

## Predict custom images

In [ ]:
# Allows users to predict custom handwritten images
def predict_image(path):
    img = load_image(path)[None, ...]
    texts, _ = decode_logits(infer_model(img, training=False))
    return texts[0]

In [ ]:
# print(predict_image("image_name_here.png"))

## Export weights

In [ ]:
import json
import numpy as np
import tensorflow as tf

In [ ]:
infer_model = tf.keras.models.load_model("/content/htr_infer_model.keras", compile=False)

arrays = []
for layer in infer_model.layers:
    for w in layer.weights:
        arr = np.asarray(w.numpy() if hasattr(w, "numpy") else w)
        if "seed_generator" in getattr(w, "path", "") or not np.issubdtype(arr.dtype, np.floating):
            continue
        arrays.append(arr.astype("<f4"))
        print(f"{layer.name:28} {tuple(arr.shape)}")

with open("/content/weights.bin", "wb") as f:
    for a in arrays:
        f.write(a.tobytes())
json.dump({"shapes": [list(a.shape) for a in arrays]}, open("/content/weights.json", "w"))

zeros = np.zeros((1, 64, 256, 1), dtype="float32")
first_row = infer_model(zeros, training=False).numpy()[0, 0].tolist()
json.dump({"logits0": first_row}, open("/content/check.json", "w"))

chars = json.load(open("/content/chars.json"))
print(len(arrays), "arrays (expect 31).  Output classes:", arrays[-1].shape[0],
      "| chars + blank:", len(chars) + 1)

In [ ]:
#  Export files into one zip
!cd /content && zip -q web_files.zip weights.bin weights.json check.json chars.json
from google.colab import files
files.download("/content/web_files.zip")